# IPCC AR6 Authors Distribution Analysis

This notebook analyzes the demographic distribution of IPCC AR6 authors using data from the Wikibase ClimateKG instance. We examine:
- **Global North vs Global South distribution**
- **Gender distribution**
- **Intersectional analysis** (gender by region)
- **Chapter-level variations**

Data source: Wikibase SPARQL endpoint (localhost:9999/bigdata/sparql)

In [10]:
# SPARQL and data processing
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import numpy as np

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set default theme for all visualizations
pio.templates.default = "plotly_white"

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [11]:
# Load author data from CSV file
# This is a temporary approach while you configure the SPARQL query with correct property IDs

import os

csv_path = os.path.join('..', 'data-xml-dtd', 'authors-ar6.csv')

if os.path.exists(csv_path):
    print("Loading author data from CSV file...")
    df_authors = pd.read_csv(csv_path)
    
    # Rename columns to match expected format
    df_authors = df_authors.rename(columns={
        'ClimateKG_Author_ID': 'author',
        'Last_Name': 'last_name',
        'First_Name': 'first_name',
        'Gender': 'gender',
        'Citizenship': 'citizenship',
        'Country_of_Residence': 'country_of_residence',
        'Chapter_QID': 'chapter',
        'Chapter': 'chapterLabel',
        'Role': 'role'
    })
    
    # Create full author name
    df_authors['authorLabel'] = df_authors['first_name'] + ' ' + df_authors['last_name']
    
    print(f"✓ Retrieved data for {df_authors['author'].nunique()} unique authors")
    print(f"Total records (including chapter contributions): {len(df_authors)}")
    print(f"\nFirst few records:")
    print(df_authors.head(10))
    print(f"\nColumns available: {list(df_authors.columns)}")
else:
    print(f"⚠ CSV file not found at: {csv_path}")
    print("\nTo use SPARQL instead, update the property IDs in the query below:")
    print("Visit http://localhost:8080/wiki/Special:ListProperties to find correct IDs")
    
    # SPARQL query (commented out - update property IDs first)
    author_query = """
    PREFIX wd: <http://localhost:8080/entity/>
    PREFIX wdt: <http://localhost:8080/prop/direct/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    
    SELECT DISTINCT 
        ?author 
        ?authorLabel 
        ?gender 
        ?citizenship 
        ?country_of_residence
        ?chapter
        ?chapterLabel
    WHERE {
        # Get authors - UPDATE property and entity IDs!
        ?author wdt:P3 wd:Q2 .  # Update P3 and Q2
        
        ?author rdfs:label ?authorLabel .
        FILTER(LANG(?authorLabel) = "en")
        
        OPTIONAL {
            ?author wdt:P7 ?genderItem .  # Update P7
            ?genderItem rdfs:label ?gender .
            FILTER(LANG(?gender) = "en")
        }
        
        OPTIONAL {
            ?author wdt:P8 ?citizenshipItem .  # Update P8
            ?citizenshipItem rdfs:label ?citizenship .
            FILTER(LANG(?citizenship) = "en")
        }
        
        OPTIONAL {
            ?author wdt:P9 ?residenceItem .  # Update P9
            ?residenceItem rdfs:label ?country_of_residence .
            FILTER(LANG(?country_of_residence) = "en")
        }
        
        OPTIONAL {
            ?author wdt:P10 ?chapter .  # Update P10
            ?chapter rdfs:label ?chapterLabel .
            FILTER(LANG(?chapterLabel) = "en")
        }
    }
    ORDER BY ?authorLabel
    """
    
    df_authors = query_wikibase(author_query)

Loading author data from CSV file...
✓ Retrieved data for 932 unique authors
Total records (including chapter contributions): 1164

First few records:
   author      last_name          first_name gender citizenship  \
0  AU0024        Aldunce             Paulina      F       Chile   
1  AU0086         Blanco             Gabriel      M   Argentina   
2  AU0109         Calvin           Katherine      F         USA   
3  AU0142         Cheung  William (wai Lung)      M      Canada   
4  AU0182       Dasgupta               Dipak      M       India   
5  AU0195         Denton              Fatima      F      Gambia   
6  AU0212  Diongue-Niang                Aïda      F     Senegal   
7  AU0216         Dodman               David      M     Jamaica   
8  AU0284     Garschagen            Matthias      M     Germany   
9  AU0286          Geden              Oliver      M     Germany   

  country_of_residence                                        Affiliation  \
0                Chile            

In [12]:
# Define Global North countries
GLOBAL_NORTH = {
    # North America
    'USA', 'United States', 'United States of America', 'Canada',
    
    # Western Europe
    'UK', 'United Kingdom', 'Great Britain', 'England', 'Scotland', 'Wales',
    'France', 'Germany', 'Italy', 'Spain', 'Portugal', 'Netherlands', 'Belgium',
    'Switzerland', 'Austria', 'Sweden', 'Norway', 'Denmark', 'Finland', 'Iceland',
    'Ireland', 'Luxembourg', 'Monaco', 'Liechtenstein',
    
    # Eastern Europe (EU members)
    'Poland', 'Czech Republic', 'Czechia', 'Hungary', 'Slovakia', 'Slovenia',
    'Croatia', 'Estonia', 'Latvia', 'Lithuania', 'Romania', 'Bulgaria',
    
    # Oceania
    'Australia', 'New Zealand',
    
    # Asia (high-income)
    'Japan', 'South Korea', 'Singapore', 'Taiwan', 'Israel',
    
    # Other
    'Greece', 'Cyprus', 'Malta'
}

def classify_region(country):
    """
    Classify a country as Global North or Global South
    """
    if pd.isna(country):
        return 'Unknown'
    
    # Normalize country name
    country = str(country).strip()
    
    if country in GLOBAL_NORTH:
        return 'Global North'
    else:
        return 'Global South'

# Apply classification
df_unique_authors['region'] = df_unique_authors['country'].apply(classify_region)

print("Regional distribution:")
print(df_unique_authors['region'].value_counts())
print()

# Show sample classifications
if df_unique_authors['country'].notna().any():
    print("Sample country classifications:")
    sample_countries = df_unique_authors[df_unique_authors['country'].notna()][['country', 'region']].drop_duplicates().head(20)
    for _, row in sample_countries.iterrows():
        print(f"  {row['country']:30s} -> {row['region']}")

Regional distribution:
region
Global North    533
Global South    399
Name: count, dtype: int64

Sample country classifications:
  Chile                          -> Global South
  Argentina                      -> Global South
  USA                            -> Global North
  Canada                         -> Global North
  India                          -> Global South
  Gambia                         -> Global South
  Senegal                        -> Global South
  Jamaica                        -> Global South
  Germany                        -> Global North
  New Zealand                    -> Global North
  UK                             -> Global North
  Australia                      -> Global North
  France                         -> Global North
  Philippines                    -> Global South
  Republic of Korea              -> Global South
  Netherlands                    -> Global North
  Ireland                        -> Global North
  South Africa                   -> Gl

## 7. Visualize Gender Distribution

Analyze and visualize gender distribution of IPCC AR6 authors.

## 9. Chapter-Level Analysis

Analyze demographic distributions at the chapter and report level to understand variations across different AR6 publications.

In [16]:
# Summary statistics
print("=" * 70)
print("IPCC AR6 AUTHOR DEMOGRAPHIC ANALYSIS - SUMMARY")
print("=" * 70)
print()

print("OVERALL STATISTICS:")
print(f"  Total unique authors: {len(df_unique_authors)}")
print(f"  Authors with country data: {df_unique_authors['country'].notna().sum()} ({df_unique_authors['country'].notna().sum() / len(df_unique_authors) * 100:.1f}%)")
print(f"  Authors with gender data: {df_unique_authors['gender'].notna().sum()} ({df_unique_authors['gender'].notna().sum() / len(df_unique_authors) * 100:.1f}%)")
print()

if (df_unique_authors['region'] != 'Unknown').any():
    print("REGIONAL DISTRIBUTION:")
    region_summary = df_unique_authors[df_unique_authors['region'] != 'Unknown']['region'].value_counts()
    for region, count in region_summary.items():
        pct = count / region_summary.sum() * 100
        print(f"  {region}: {count} authors ({pct:.1f}%)")
    print()

if df_unique_authors['gender'].notna().any():
    print("GENDER DISTRIBUTION:")
    gender_summary = df_unique_authors[df_unique_authors['gender'].notna()]['gender'].value_counts()
    for gender, count in gender_summary.items():
        pct = count / gender_summary.sum() * 100
        print(f"  {gender}: {count} authors ({pct:.1f}%)")
    print()

if len(df_gender_region) > 0:
    print("GENDER BY REGION:")
    for region in df_gender_region['region'].unique():
        region_data = df_gender_region[df_gender_region['region'] == region]
        gender_dist = region_data['gender'].value_counts()
        print(f"  {region}:")
        for gender, count in gender_dist.items():
            pct = count / len(region_data) * 100
            print(f"    {gender}: {count} ({pct:.1f}%)")
    print()

if len(df_chapters) > 0:
    print("CHAPTER CONTRIBUTIONS:")
    print(f"  Total contributions: {len(df_chapters)}")
    print(f"  Unique chapters: {df_chapters['chapter'].nunique()}")
    avg_contrib = len(df_chapters) / df_authors['author'].nunique()
    print(f"  Average contributions per author: {avg_contrib:.1f}")
    print()

print("=" * 70)
print("✓ Analysis complete")
print()
print("Key Findings:")
print("- Regional representation shows distribution between Global North and Global South")
print("- Gender analysis reveals demographic composition of author teams")
print("- Intersectional analysis identifies patterns across regions and gender")
print("- Chapter-level variation demonstrates diversity across different AR6 reports")

IPCC AR6 AUTHOR DEMOGRAPHIC ANALYSIS - SUMMARY

OVERALL STATISTICS:
  Total unique authors: 932
  Authors with country data: 932 (100.0%)
  Authors with gender data: 932 (100.0%)

REGIONAL DISTRIBUTION:
  Global North: 533 authors (57.2%)
  Global South: 399 authors (42.8%)

GENDER DISTRIBUTION:
  M: 635 authors (68.1%)
  F: 297 authors (31.9%)

GENDER BY REGION:
  Global South:
    M: 280 (70.2%)
    F: 119 (29.8%)
  Global North:
    M: 355 (66.6%)
    F: 178 (33.4%)

CHAPTER CONTRIBUTIONS:
  Total contributions: 1164
  Unique chapters: 75
  Average contributions per author: 1.2

✓ Analysis complete

Key Findings:
- Regional representation shows distribution between Global North and Global South
- Gender analysis reveals demographic composition of author teams
- Intersectional analysis identifies patterns across regions and gender
- Chapter-level variation demonstrates diversity across different AR6 reports


## Notes and Next Steps

### Observations
- This analysis provides insights into the demographic composition of IPCC AR6 author teams
- The Global North/South classification helps identify regional representation patterns
- Gender distribution analysis reveals equity considerations in IPCC authorship
- Combined analysis shows intersectional patterns between gender and region

### Potential Extensions
1. **Temporal analysis**: Compare demographics across different IPCC assessment reports (AR5 vs AR6)
2. **Role analysis**: Examine demographics by authorship role (Lead Author, Review Editor, etc.)
3. **Affiliation analysis**: Analyze institutional diversity and academic vs non-academic representation
4. **Correlation studies**: Explore relationships between demographics and contribution patterns
5. **Interactive dashboard**: Build a Dash or Streamlit app for dynamic data exploration
6. **Geographic mapping**: Create world maps showing author distribution by country
7. **Statistical testing**: Perform chi-square tests for significant differences between groups

### Data Quality Considerations
- **Missing data**: Some authors may lack country or gender information in the dataset
- **Classification**: Global North/South is a simplification; consider more nuanced classifications (e.g., HDI, income levels)
- **Gender**: Binary classification may not capture full diversity; update when more inclusive data becomes available
- **Country mapping**: Standardize country names carefully (e.g., "USA" vs "United States")
- **Dual citizenship**: Authors with multiple citizenships may only have one recorded

### SPARQL Query Customization
If the SPARQL query needs adjustment for your Wikibase schema:
- Update property IDs (P3, P7, P8, P9, P10) to match your instance
- Modify entity IDs (Q2 for Person) as needed
- Add additional properties for enriched analysis (e.g., institutional affiliation, seniority)
- Visit http://localhost:8080/wiki/Special:ListProperties to see all available properties

### Required Python Packages
Install required packages in your environment:
```bash
pip install SPARQLWrapper pandas plotly numpy
```

### Environment
- **Wikibase**: http://localhost:8080
- **SPARQL endpoint**: http://localhost:9999/bigdata/sparql
- **Python environment**: .venv in C:\Wikibase
- **Docker compose**: `docker compose up -d` to start services

---

**Analysis Notebook Created**: June 2026  
**Data Source**: IPCC AR6 Authors (authors-ar6.xml)  
**Wikibase Instance**: ClimateKG LOCAL

## 10. Summary Statistics and Key Findings

Compile key metrics and insights from the demographic analysis.

In [ ]:
# Gender distribution across chapters (for chapters with most contributions)
if len(df_chapters) > 0 and 'gender' in df_chapters.columns:
    # Get top 10 chapters
    top_chapters = df_chapters['chapterLabel'].value_counts().head(10).index
    df_top_chapters = df_chapters[df_chapters['chapterLabel'].isin(top_chapters)].copy()
    
    # Create unique author-chapter combinations with demographics
    df_chapter_demo = df_top_chapters.drop_duplicates(subset=['author', 'chapter'])
    
    if df_chapter_demo['gender'].notna().any():
        # Count by chapter and gender
        chapter_gender = df_chapter_demo[df_chapter_demo['gender'].notna()].groupby(
            ['chapterLabel', 'gender']
        ).size().reset_index(name='count')
        
        fig = px.bar(
            chapter_gender,
            x='chapterLabel',
            y='count',
            color='gender',
            title='Gender Distribution in Top 10 Chapters',
            labels={'count': 'Number of Authors', 'chapterLabel': 'Chapter', 'gender': 'Gender'},
            color_discrete_map={'Female': '#9b59b6', 'Male': '#3498db', 'F': '#9b59b6', 'M': '#3498db'},
            barmode='group'
        )
        
        fig.update_layout(
            height=600,
            xaxis={'tickangle': -45},
            showlegend=True
        )
        
        fig.show()

In [14]:
# Analyze contributions by chapter
df_chapters = df_authors[df_authors['chapter'].notna()].copy()

if len(df_chapters) > 0:
    print(f"Total chapter contributions: {len(df_chapters)}")
    print(f"Unique chapters: {df_chapters['chapter'].nunique()}")
    print()
    
    # Top chapters by author count
    chapter_counts = df_chapters['chapterLabel'].value_counts().head(15)
    print("Top 15 chapters by author contributions:")
    print(chapter_counts)
    print()
    
    # Visualize
    fig = px.bar(
        x=chapter_counts.values,
        y=chapter_counts.index,
        orientation='h',
        title='Top 15 Chapters by Number of Author Contributions',
        labels={'x': 'Number of Author Contributions', 'y': 'Chapter'},
        text=chapter_counts.values
    )
    
    fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
    fig.update_traces(textposition='outside', marker_color='#16a085')
    
    fig.show()
else:
    print("⚠ No chapter contribution data available")

Total chapter contributions: 1164
Unique chapters: 75

Top 15 chapters by author contributions:
chapterLabel
SPM                                                                                              30
LR                                                                                               30
Chapter 1: Framing and Context                                                                   29
Chapter 9: Ocean, cryosphere, and sea level change                                               21
Chapter 3: Impacts of 1.5°C global warming on natural and human systems                          21
Chapter 4: Strengthening and implementing the global response to the threat of climate change    21
ATLAS                                                                                            19
Chapter 3: Mitigation pathways compatible with long-term goals                                   19
Chapter 11: Weather and climate extreme events in a changing climate                       

In [15]:
# Gender distribution by region
df_gender_region = df_unique_authors[
    (df_unique_authors['gender'].notna()) & 
    (df_unique_authors['region'] != 'Unknown')
].copy()

if len(df_gender_region) > 0:
    # Cross-tabulation
    crosstab = pd.crosstab(df_gender_region['region'], df_gender_region['gender'])
    print("Gender distribution by region (counts):")
    print(crosstab)
    print()
    
    # Calculate percentages
    crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100
    print("Percentage within each region:")
    print(crosstab_pct.round(1))
    print()
    
    # Prepare data for visualization
    gender_region_counts = df_gender_region.groupby(['region', 'gender']).size().reset_index(name='count')
    
    # Grouped bar chart
    fig = px.bar(
        gender_region_counts,
        x='region',
        y='count',
        color='gender',
        title='Gender Distribution by Global Region (Grouped)',
        labels={'count': 'Number of Authors', 'region': 'Region', 'gender': 'Gender'},
        color_discrete_map={'Female': '#9b59b6', 'Male': '#3498db', 'F': '#9b59b6', 'M': '#3498db'},
        text='count',
        barmode='group'
    )
    
    fig.update_layout(height=500)
    fig.update_traces(textposition='outside')
    
    fig.show()
    
    # Percentage stacked bar chart
    gender_region_pct = gender_region_counts.copy()
    total_by_region = gender_region_pct.groupby('region')['count'].transform('sum')
    gender_region_pct['percentage'] = (gender_region_pct['count'] / total_by_region * 100).round(1)
    
    fig2 = px.bar(
        gender_region_pct,
        x='region',
        y='percentage',
        color='gender',
        title='Gender Distribution by Region (Percentage, Stacked)',
        labels={'percentage': 'Percentage', 'region': 'Region', 'gender': 'Gender'},
        color_discrete_map={'Female': '#9b59b6', 'Male': '#3498db', 'F': '#9b59b6', 'M': '#3498db'},
        text='percentage',
        barmode='stack'
    )
    
    fig2.update_layout(height=500, yaxis={'range': [0, 100]})
    fig2.update_traces(texttemplate='%{text:.1f}%', textposition='inside')
    
    fig2.show()
else:
    print("⚠ Insufficient data for combined gender/region analysis")

Gender distribution by region (counts):
gender          F    M
region                
Global North  178  355
Global South  119  280

Percentage within each region:
gender           F     M
region                  
Global North  33.4  66.6
Global South  29.8  70.2



## 8. Combined Analysis: Gender by Region

Examine the intersection of gender and regional distribution to understand demographic patterns across Global North and Global South.

In [ ]:
# Gender distribution analysis
df_gender = df_unique_authors[df_unique_authors['gender'].notna()].copy()

if len(df_gender) > 0:
    gender_counts = df_gender['gender'].value_counts().reset_index()
    gender_counts.columns = ['Gender', 'Count']
    gender_counts['Percentage'] = (gender_counts['Count'] / gender_counts['Count'].sum() * 100).round(1)
    
    print("Gender distribution:")
    print(gender_counts)
    print()
    
    # Create visualizations
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Author Count by Gender', 'Percentage Distribution'),
        specs=[[{"type": "bar"}, {"type": "pie"}]]
    )
    
    # Bar chart
    colors_gender = {'Female': '#9b59b6', 'Male': '#3498db', 'F': '#9b59b6', 'M': '#3498db'}
    for gender in gender_counts['Gender']:
        data = gender_counts[gender_counts['Gender'] == gender]
        fig.add_trace(
            go.Bar(
                x=data['Gender'],
                y=data['Count'],
                name=gender,
                marker_color=colors_gender.get(gender, '#95a5a6'),
                text=data['Count'],
                textposition='outside',
                showlegend=False
            ),
            row=1, col=1
        )
    
    # Pie chart
    fig.add_trace(
        go.Pie(
            labels=gender_counts['Gender'],
            values=gender_counts['Count'],
            marker=dict(colors=[colors_gender.get(g, '#95a5a6') for g in gender_counts['Gender']]),
            textinfo='label+percent',
            textposition='inside'
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title_text="IPCC AR6 Authors: Gender Distribution",
        height=500,
        showlegend=False
    )
    
    fig.show()
    
    # Calculate gender ratio
    if len(gender_counts) == 2:
        ratio = gender_counts.iloc[0]['Count'] / gender_counts.iloc[1]['Count']
        print(f"\nGender ratio: {gender_counts.iloc[0]['Gender']}:{gender_counts.iloc[1]['Gender']} = {ratio:.2f}:1")
else:
    print("⚠ No gender data available for visualization")

In [ ]:
# Country-level bar chart (top countries)
if 'country' in df_unique_authors.columns and df_unique_authors['country'].notna().any():
    df_country = df_unique_authors[df_unique_authors['country'].notna()].copy()
    country_counts = df_country.groupby(['country', 'region']).size().reset_index(name='count')
    country_counts = country_counts.sort_values('count', ascending=False).head(20)
    
    fig = px.bar(
        country_counts,
        x='count',
        y='country',
        color='region',
        orientation='h',
        title='Top 20 Countries by Author Count',
        labels={'count': 'Number of Authors', 'country': 'Country', 'region': 'Region'},
        color_discrete_map={'Global North': '#3498db', 'Global South': '#e74c3c', 'Unknown': '#95a5a6'},
        text='count'
    )
    
    fig.update_layout(
        height=600,
        yaxis={'categoryorder': 'total ascending'},
        showlegend=True
    )
    
    fig.update_traces(textposition='outside')
    
    fig.show()

In [ ]:
# Filter out Unknown region for visualization
df_regional = df_unique_authors[df_unique_authors['region'] != 'Unknown'].copy()

if len(df_regional) > 0:
    # Count by region
    region_counts = df_regional['region'].value_counts().reset_index()
    region_counts.columns = ['Region', 'Count']
    region_counts['Percentage'] = (region_counts['Count'] / region_counts['Count'].sum() * 100).round(1)
    
    print(f"Authors by region (excluding {(df_unique_authors['region'] == 'Unknown').sum()} with unknown country):")
    print(region_counts)
    print()
    
    # Create visualizations
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Author Distribution by Region', 'Percentage Distribution'),
        specs=[[{"type": "bar"}, {"type": "pie"}]]
    )
    
    # Bar chart
    colors = {'Global North': '#3498db', 'Global South': '#e74c3c'}
    for region in region_counts['Region']:
        data = region_counts[region_counts['Region'] == region]
        fig.add_trace(
            go.Bar(
                x=data['Region'],
                y=data['Count'],
                name=region,
                marker_color=colors.get(region, '#95a5a6'),
                text=data['Count'],
                textposition='outside',
                showlegend=False
            ),
            row=1, col=1
        )
    
    # Pie chart
    fig.add_trace(
        go.Pie(
            labels=region_counts['Region'],
            values=region_counts['Count'],
            marker=dict(colors=[colors.get(r, '#95a5a6') for r in region_counts['Region']]),
            textinfo='label+percent',
            textposition='inside'
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title_text="IPCC AR6 Authors: Global North vs Global South Distribution",
        height=500,
        showlegend=False
    )
    
    fig.show()
else:
    print("⚠ No regional data available for visualization")

### Regional Distribution by AR6 Report Series

Analyze how Global North vs Global South representation varies across different AR6 reports:

- **SYR**: Synthesis Report
- **WGI**: Working Group I (Physical Science Basis)
- **WGII**: Working Group II (Impacts, Adaptation and Vulnerability)
- **WGIII**: Working Group III (Mitigation of Climate Change)
- **SR1.5**: Special Report on Global Warming of 1.5°C
- **SRCCL**: Special Report on Climate Change and Land
- **SROCC**: Special Report on the Ocean and Cryosphere

In [17]:
# Regional distribution by Report/Series
# Combine unique authors with their report contributions

# Get author-report combinations with regions
df_author_reports = df_authors[['author', 'Report']].copy()
df_author_reports = df_author_reports.merge(
    df_unique_authors[['author', 'region']], 
    on='author', 
    how='left'
)

# Remove Unknown regions
df_author_reports = df_author_reports[df_author_reports['region'] != 'Unknown']

# Count unique authors per report and region
report_region_counts = df_author_reports.drop_duplicates(subset=['author', 'Report']).groupby(
    ['Report', 'region']
).size().reset_index(name='count')

print("Author distribution by Report and Region:")
print(report_region_counts.pivot(index='Report', columns='region', values='count').fillna(0))
print()

# Create grouped bar chart by Report
fig = px.bar(
    report_region_counts,
    x='Report',
    y='count',
    color='region',
    title='Global North vs Global South Distribution by AR6 Report',
    labels={'count': 'Number of Unique Authors', 'Report': 'AR6 Report', 'region': 'Region'},
    color_discrete_map={'Global North': '#3498db', 'Global South': '#e74c3c'},
    text='count',
    barmode='group'
)

fig.update_layout(
    height=500,
    xaxis={'categoryorder': 'total descending'},
    showlegend=True,
    legend=dict(title="Region")
)

fig.update_traces(textposition='outside')

fig.show()

# Create percentage stacked bar chart by Report
report_region_pct = report_region_counts.copy()
total_by_report = report_region_pct.groupby('Report')['count'].transform('sum')
report_region_pct['percentage'] = (report_region_pct['count'] / total_by_report * 100).round(1)

fig2 = px.bar(
    report_region_pct,
    x='Report',
    y='percentage',
    color='region',
    title='Global North vs Global South Distribution by AR6 Report (Percentage)',
    labels={'percentage': 'Percentage of Authors (%)', 'Report': 'AR6 Report', 'region': 'Region'},
    color_discrete_map={'Global North': '#3498db', 'Global South': '#e74c3c'},
    text='percentage',
    barmode='stack'
)

fig2.update_layout(
    height=500,
    xaxis={'categoryorder': 'total descending'},
    yaxis={'range': [0, 100]},
    showlegend=True,
    legend=dict(title="Region")
)

fig2.update_traces(texttemplate='%{text:.1f}%', textposition='inside')

fig2.show()

# Overall AR6 totals (all reports combined)
overall_counts = df_unique_authors[df_unique_authors['region'] != 'Unknown']['region'].value_counts()
print(f"\nOverall AR6 Distribution (All Reports):")
print(f"  Global North: {overall_counts['Global North']} authors ({overall_counts['Global North']/overall_counts.sum()*100:.1f}%)")
print(f"  Global South: {overall_counts['Global South']} authors ({overall_counts['Global South']/overall_counts.sum()*100:.1f}%)")
print(f"  Total: {overall_counts.sum()} authors")

Author distribution by Report and Region:
region  Global North  Global South
Report                            
SR1.5             45            46
SRCCL             47            60
SROCC             69            34
SYR               15            15
WGI              139            95
WGII             156           114
WGIII            129           110




Overall AR6 Distribution (All Reports):
  Global North: 533 authors (57.2%)
  Global South: 399 authors (42.8%)
  Total: 932 authors


### Overall AR6 Regional Distribution Summary

Comprehensive view of all unique authors across the entire AR6 assessment cycle.

In [18]:
# Overall AR6 Distribution Visualization (All Unique Authors)
# This summarizes all unique authors regardless of how many reports they contributed to

overall_region_data = df_unique_authors[df_unique_authors['region'] != 'Unknown']['region'].value_counts().reset_index()
overall_region_data.columns = ['Region', 'Count']
overall_region_data['Percentage'] = (overall_region_data['Count'] / overall_region_data['Count'].sum() * 100).round(1)

print("=" * 70)
print("OVERALL AR6 REGIONAL DISTRIBUTION")
print("=" * 70)
for _, row in overall_region_data.iterrows():
    print(f"  {row['Region']}: {row['Count']} authors ({row['Percentage']}%)")
print(f"  Total: {overall_region_data['Count'].sum()} authors")
print("=" * 70)
print()

# Create combined visualization with bar and pie charts
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Total Authors by Region', 'Percentage Distribution'),
    specs=[[{"type": "bar"}, {"type": "pie"}]],
    horizontal_spacing=0.15
)

# Bar chart
colors = {'Global North': '#3498db', 'Global South': '#e74c3c'}
for region in overall_region_data['Region']:
    data = overall_region_data[overall_region_data['Region'] == region]
    fig.add_trace(
        go.Bar(
            x=data['Region'],
            y=data['Count'],
            name=region,
            marker_color=colors.get(region, '#95a5a6'),
            text=data['Count'],
            textposition='outside',
            showlegend=False,
            hovertemplate='<b>%{x}</b><br>Authors: %{y}<br><extra></extra>'
        ),
        row=1, col=1
    )

# Pie chart
fig.add_trace(
    go.Pie(
        labels=overall_region_data['Region'],
        values=overall_region_data['Count'],
        marker=dict(colors=[colors.get(r, '#95a5a6') for r in overall_region_data['Region']]),
        textinfo='label+percent',
        textposition='inside',
        hovertemplate='<b>%{label}</b><br>Authors: %{value}<br>Percentage: %{percent}<br><extra></extra>'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Overall AR6 Authors: Global North vs Global South Distribution<br><sub>Based on 932 Unique Authors Across All Reports</sub>",
    height=500,
    showlegend=False
)

fig.update_xaxes(title_text="Region", row=1, col=1)
fig.update_yaxes(title_text="Number of Authors", row=1, col=1)

fig.show()

# Summary statistics
total_authors = overall_region_data['Count'].sum()
north_count = overall_region_data[overall_region_data['Region'] == 'Global North']['Count'].values[0]
south_count = overall_region_data[overall_region_data['Region'] == 'Global South']['Count'].values[0]

print(f"Key Finding:")
print(f"The IPCC AR6 assessment shows a {north_count/south_count:.2f}:1 ratio of Global North to Global South authors.")
print(f"Global North authors represent {north_count/total_authors*100:.1f}% of all contributors.")

OVERALL AR6 REGIONAL DISTRIBUTION
  Global North: 533 authors (57.2%)
  Global South: 399 authors (42.8%)
  Total: 932 authors



Key Finding:
The IPCC AR6 assessment shows a 1.34:1 ratio of Global North to Global South authors.
Global North authors represent 57.2% of all contributors.


## 6. Visualize Global North vs Global South Distribution

Create interactive Plotly visualizations showing regional distribution of authors.

## 5. Classify Countries by Global North/South

Create a classification function based on standard definitions of Global North and Global South.

**Global North**: Generally includes high-income industrialized countries (USA, Canada, most of Europe, Japan, Australia, New Zealand, etc.)

**Global South**: Generally includes low- and middle-income developing countries (most of Africa, Asia, Latin America, and the Caribbean)

In [ ]:
# Create a copy for processing
df = df_authors.copy()

# Check for missing data
print("Missing data analysis:")
print(df.isnull().sum())
print()

# Use country of residence if citizenship is missing, and vice versa
if 'citizenship' not in df.columns:
    df['citizenship'] = None
if 'country_of_residence' not in df.columns:
    df['country_of_residence'] = None

df['country'] = df['citizenship'].fillna(df['country_of_residence'])

# Create a unique author dataset (one row per author)
df_unique_authors = df.drop_duplicates(subset=['author']).copy()

print(f"✓ Unique authors dataset created: {len(df_unique_authors)} authors")
print(f"Authors with gender data: {df_unique_authors['gender'].notna().sum()}")
print(f"Authors with country data: {df_unique_authors['country'].notna().sum()}")
print()

# Display gender distribution
if 'gender' in df_unique_authors.columns and df_unique_authors['gender'].notna().any():
    print("Gender distribution:")
    print(df_unique_authors['gender'].value_counts())
    print()

# Display top countries
if 'country' in df_unique_authors.columns and df_unique_authors['country'].notna().any():
    print("Top 10 countries by author count:")
    print(df_unique_authors['country'].value_counts().head(10))

## 4. Data Preprocessing and Cleaning

Clean the data, handle missing values, and prepare for analysis.

## 3. Query Author Data from Wikibase

Fetch author information including gender, citizenship, country of residence, and chapter contributions.

**Note**: The SPARQL query below uses example property IDs. You may need to adjust them to match your Wikibase schema. Check your instance at http://localhost:8080/wiki/Special:ListProperties to find the correct property numbers.

In [ ]:
# Wikibase SPARQL endpoint configuration
SPARQL_ENDPOINT = "http://localhost:9999/bigdata/sparql"

def query_wikibase(sparql_query):
    """
    Execute a SPARQL query against the Wikibase endpoint
    Returns results as a pandas DataFrame
    """
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setQuery(sparql_query)
    sparql.setReturnFormat(JSON)
    
    try:
        results = sparql.query().convert()
        
        # Convert to pandas DataFrame
        bindings = results["results"]["bindings"]
        if not bindings:
            return pd.DataFrame()
        
        data = []
        for binding in bindings:
            row = {key: binding[key]["value"] for key in binding.keys()}
            data.append(row)
        
        df = pd.DataFrame(data)
        print(f"✓ Query returned {len(df)} results")
        return df
    
    except Exception as e:
        print(f"✗ Query failed: {e}")
        return pd.DataFrame()

print("✓ SPARQL endpoint configured")
print(f"Endpoint: {SPARQL_ENDPOINT}")

## 2. Configure Wikibase SPARQL Endpoint

Set up connection to the local Wikibase SPARQL endpoint and create helper functions.

## 1. Setup and Import Libraries
Import necessary Python libraries for SPARQL queries, data manipulation, and visualization.